# 7. Route C — Die Kontraktions-Eichung

## Worum es geht

Das Problem von Route B ist das der Companion-Operator $E$ die Gedächtnissumme nach $K$ Termen abschneidet. Alles, was länger als $K\Delta t$ zurückliegt, ist verloren (der Rest-Term $R_n$ aus der Tabelle in dem Abschnitt *Der Companionpropagator*) und die Dynamik wird ungenauer je weiter die Simulation fortschreitet. Route C löst das, idem es keinen Companiontensor, der nur für ein kleines Zeitfenster gilt und keine ADOs enthält, aus dem HEOM-Propagator berechnet, sondern den HEOM-Propagator, der die gesamte Dynamik enthält, selbst nimmt und ihn anstatt $E$ auf das Grid gibt:

$$P = \exp(\mathcal{L}_{\mathrm{HEOM}}\,\Delta t) \in \mathbb{C}^{\mathcal{D}_{\mathrm{tot}}\times\mathcal{D}_{\mathrm{tot}}},\qquad \mathcal{D}_{\mathrm{tot}} = \mathcal{N}_{\mathrm{ADOs}}\cdot d^2 .$$

Die Operatornorm von $E$ war mit $\sqrt{2}$ beschränkt. Die Operatornorm von $P$ ist jedoch leider viel größer: $\Vert P\Vert_2 = 32{,}39$. Nach Proposition 13 multiplizieren sich die Erfolgswahrscheinlichkeiten, und $32{,}39^{-200}$ ist ungefähr $10^{-301}$. Man kann jedoch eine geschickte Koordinatentransformation machen, sodass $\Vert P\Vert_2 \approx 1$. Danach kann man dann eine Arnoldicompression machen da $P$ wegen den ganzen ADOs ja eine sehr große Matrix ist (viel größer als $E$). Im Fall von $P$ kann man das $m$ jedoch so groß wählen wie man will ohne dass die Norm $\Vert P\Vert_21$ wieder zu wachsen beginnt.

## Scaled HEOM

Die normale HEOM formel lautet:
$$\frac{d}{dt}\hat{\sigma}_{\mathbf{n}}(t) = \underbrace{ -\frac{i}{\hbar}\mathcal{L}_S \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}}(t) }_{\text{Term 1: Eigendynamik \& Zerfall}} + \underbrace{ \sum_{j,k} \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}(t) }_{\text{Term 2: Kopplung nach Oben}} + \underbrace{ \sum_{j,k} n_{jk} \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}(t) }_{\text{Term 3: Kopplung nach Unten}}$$

Wir wollen das jetzt in scaled HEOM umwandeln, dass wie folgt aussieht:
$$\frac{d}{dt}\hat{\sigma}_{\mathbf{n}}(t) = \underbrace{ -\frac{i}{\hbar}\mathcal{L}_S \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}} }_{\text{Unskalierter Diagonalterm}} + \underbrace{ \sum_{j,k} \sqrt{(n_{jk}+1)\frac{|a_{jk}|}{\hbar}} \, \phi_j \hat{\sigma}_{\mathbf{n}_{jk,+}} }_{\text{Skaliert mit scaling\_up}} + \underbrace{ \sum_{j,k} \sqrt{\frac{n_{jk}}{|a_{jk}|/\hbar}} \, \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk,-}} }_{\text{Skaliert mit scaling\_down}}$$

Dazu definieren wir $$\tilde{\sigma}_{\mathbf{n}}(t) = S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}(t) \quad \text{mit} \quad S_{\mathbf{n}} = \sqrt{ \prod_{j,k} n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} }$$
und berechnen $$\frac{d}{dt} \Big(\hat{\sigma}_{\mathbf{n}} \Big) = \frac{1}{S_n}\frac{d}{dt} \Big( S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}} \Big) = \frac{1}{S_n} \text{Term 1}(S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}) + \frac{1}{S_n}\text{Term 2}(S_{\mathbf{n}_{jk+}} \hat{\sigma}_{\mathbf{n}_{jk+}}) + \frac{1}{S_n} \text{Term 3}(S_{\mathbf{n}_{jk-}} \hat{\sigma}_{\mathbf{n}_{jk-}})$$

### Term 1
$$ \frac{1}{S_n} \text{Term 1}(S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}})= \frac{1}{S_n} \left( -\frac{i}{\hbar}[H, S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}}] + \sum_{j,k} \left( \tilde{\phi}_j\tilde{\theta}_{j,0} - n_{jk}\gamma_{jk} \right) S_{\mathbf{n}} \hat{\sigma}_{\mathbf{n}} \right) = \frac{1}{S_n} S_{\mathbf{n}} \left( -\frac{i}{\hbar}[H, \hat{\sigma}_{\mathbf{n}}] + \sum_{j,k} \left( \tilde{\phi}_j\tilde{\theta}_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}} \right) = \text{Term 1}(\hat{\sigma}_{\mathbf{n}})$$

### Term 2
$$\begin{align*}
\frac{1}{S_{\mathbf{n}}} \text{Term 2}\left(S_{\mathbf{n}_{jk+}} \hat{\sigma}_{\mathbf{n}_{jk+}}\right) &= \frac{1}{S_{\mathbf{n}}} \sum_{j,k} \tilde{\phi}_j \left( S_{\mathbf{n}_{jk+}} \hat{\sigma}_{\mathbf{n}_{jk+}} \right) = \sum_{j,k} \left( \frac{S_{\mathbf{n}_{jk+}}}{S_{\mathbf{n}}} \right) \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}} = \sum_{j,k} \sqrt{ \frac{ (n_{jk}+1)! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}+1} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} }{ n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} } } \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}} \\
&= \sum_{j,k} \sqrt{ \frac{ (n_{jk}+1) \cdot n_{jk}! \cdot \left( \frac{|a_{jk}|}{\hbar} \right) \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} }{ n_{jk}! \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} } } \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}} = \sum_{j,k} \sqrt{ (n_{jk}+1) \frac{|a_{jk}|}{\hbar} } \, \tilde{\phi}_j \hat{\sigma}_{\mathbf{n}_{jk+}}
\end{align*}$$

### Term 3

$$\begin{align*}
\frac{1}{S_{\mathbf{n}}} \text{Term 3}\left(S_{\mathbf{n}_{jk-}} \hat{\sigma}_{\mathbf{n}_{jk-}}\right) &= \frac{1}{S_{\mathbf{n}}} \sum_{j,k} n_{jk} \tilde{\theta}_{j,k} \left( S_{\mathbf{n}_{jk-}} \hat{\sigma}_{\mathbf{n}_{jk-}} \right) = \sum_{j,k} n_{jk} \left( \frac{S_{\mathbf{n}_{jk-}}}{S_{\mathbf{n}}} \right) \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} n_{jk} \sqrt{ \frac{ (n_{jk}-1)! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}-1} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} }{ n_{jk}! \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}} \prod_{(j',k') \neq (j,k)} n_{j'k'}! \left( \frac{|a_{j'k'}|}{\hbar} \right)^{n_{j'k'}} } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} \\
&= \sum_{j,k} n_{jk} \sqrt{ \frac{ (n_{jk}-1)! \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}-1} }{ n_{jk} \cdot (n_{jk}-1)! \cdot \left( \frac{|a_{jk}|}{\hbar} \right) \cdot \left( \frac{|a_{jk}|}{\hbar} \right)^{n_{jk}-1} } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} n_{jk} \sqrt{ \frac{ 1 }{ n_{jk} \left( \frac{|a_{jk}|}{\hbar} \right) } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} = \sum_{j,k} \sqrt{ n_{jk}^2 \cdot \frac{ 1 }{ n_{jk} \left( \frac{|a_{jk}|}{\hbar} \right) } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}} \\
&= \sum_{j,k} \sqrt{ \frac{ n_{jk} }{ |a_{jk}| / \hbar } } \tilde{\theta}_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}
\end{align*}$$

### Intuition hinter der Skalierung
Mit $\phi_j = i V_j^\times$ und $\theta_{j,k\neq0} = i a_{jk} V_j^\times$ wobei $a_{jk} = \frac{4 \lambda \gamma \nu_k}{\beta (\nu_k^2 - \gamma^2)}$ und
$$\frac{d}{dt}\hat{\sigma}_{\mathbf{n}}(t) = -\frac{i}{\hbar}\mathcal{L}_S \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \left( \phi_j \theta_{j,0} - n_{jk}\gamma_{jk} \right) \hat{\sigma}_{\mathbf{n}}(t) + \sum_{j,k} \big( \phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}(t) + n_{jk} \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}(t) \big)$$
kann man sehen dass der Term $\phi_j \hat{\sigma}_{\mathbf{n}_{jk+}}(t)$ keinen Vorfaktor enthält wohingegen der Term $n_{jk} \theta_{j,k} \hat{\sigma}_{\mathbf{n}_{jk-}}(t)$ den Forfaktor $(n_{jk}+1) \frac{a_{jk}}{\hbar}$ enthält. Dieses Ungleichgewicht der beiden Terme führt in der Matrix der HEOM gleichung (7.1) zu sehr verschiedenen Eigenwerten und erzeugt somit ein steifes Problem. Dieses lässt sich von numerischen Integratoren nur mit sehr kleinen Zeitschritten lösen und ist daher sehr teuer. Selbst wenn die Werte für $n_{jk}$ nur klein sind, pflanzen si sich durch die Hierarchie wie faktorielle fort was zu der großen Ungleichheit der Terme führt. Deshalb ist es besser ein scaling einzuführen um die Matrix zu Preconditionen und die eigenwerte besser aneinander anzupassen.

### 1. Transformation des HEOM-Generators

Der Gesamtzustandsvektor aller ADOs sei $\vec{\Sigma}(t)$, und die Dynamik ist gegeben durch:
$$\frac{\mathrm{d}}{\mathrm{d}t} \vec{\Sigma}(t) = \mathcal{L}_{\mathrm{HEOM}} \vec{\Sigma}(t) \tag{7.1}$$

Die umskalierte Koordinate lautet $\widetilde{\Sigma}_i(t) = \Sigma_i(t) / s_i$, wobei $s_i > 0$ die Einträge des Skalierungsvektors `scale` sind. In Matrixschreibweise mit der Diagonalmatrix $D = \operatorname{diag}(s_1, \dots, s_N)$ gilt:
$\widetilde{\vec{\Sigma}}(t) = D^{-1} \vec{\Sigma}(t)$ und deswegen $\vec{\Sigma}(t) = D\, \widetilde{\vec{\Sigma}}(t)$. Einsetzen in die Bewegungsgleichung (7.1) liefert:
$$\frac{\mathrm{d}}{\mathrm{d}t} \big(D\, \widetilde{\vec{\Sigma}}(t)\big) = \mathcal{L}_{\mathrm{HEOM}} \big(D\, \widetilde{\vec{\Sigma}}(t)\big) \qquad \implies \qquad\frac{\mathrm{d}}{\mathrm{d}t} \widetilde{\vec{\Sigma}}(t) = \underbrace{\big(D^{-1} \mathcal{L}_{\mathrm{HEOM}} D\big)}_{\widetilde{\mathcal{L}}_{\mathrm{HEOM}}} \widetilde{\vec{\Sigma}}(t)$$

Der transformierte Generator ist somit eine Ähnlichkeitstransformation: $\widetilde{\mathcal{L}}_{\mathrm{HEOM}} = D^{-1} \mathcal{L}_{\mathrm{HEOM}} D$. Da $D$ eine Diagonalmatrix ist, gilt $D_{jj} = s_j$ und $(D^{-1})_{ii} = \frac{1}{s_i}$. Für das Matrixelement $(i, j)$ des transformierten Operators folgt direkt:
$$\big(\widetilde{\mathcal{L}}_{\mathrm{HEOM}}\big)_{ij} = \big(D^{-1} \mathcal{L}_{\mathrm{HEOM}} D\big)_{ij} = (D^{-1})_{ii} \cdot (\mathcal{L}_{\mathrm{HEOM}})_{ij} \cdot D_{jj} = \frac{1}{s_i} \cdot (\mathcal{L}_{\mathrm{HEOM}})_{ij} \cdot s_j = (\mathcal{L}_{\mathrm{HEOM}})_{ij} \cdot \frac{s_j}{s_i}$$

Der Ausdruck `(scale[None, :] / scale[:, None])`ist ein äußeres Produkt und spannt eine Matrix der Dimension $(N \times N)$ mit genau den Einträgen $\frac{s_j}{s_i}$ an Position $(i, j)$ auf. Die wird dann elementweise auf $A$ multipliziert: `A = A * (scale[None, :] / scale[:, None])`

## Der HEOM Propagator $P$ bildet eine Halbgruppe

Im nicht-Markovschen Fall ist das Bad kein passiver Zuschauer: Es merkt sich Wechselwirkungen und tauscht Energie sowie Korrelationen mit dem System aus. Die Standard-HEOM (Hierarchical Equations of Motion) löst dieses Problem, indem sie den Zustandsraum erweitert. Wir fassen die vektorisierte Systemdichtematrix $\text{vec}(\rho_S)$ und das dynamische Gedächtnis des Bades in einem einzigen großen Super-Vektor $\vec{\Sigma}(t)$ zusammen:

$$\vec{\Sigma}(t) = \begin{pmatrix} \text{vec}(\rho_S(t)) \\ \text{vec}(\mathrm{ADO}_1(t)) \\ \vdots \\ \text{vec}(\mathrm{ADO}_{\mathcal{N}_{\mathrm{ADOs}}-1}(t)) \end{pmatrix} \in \mathbb{C}^{\mathcal{D}_{\mathrm{tot}}}$$

* **$\text{vec}(\rho_S)$ (oberster Block):** Die vektorisierte Systemdichtematrix der Dimension $d^2$ (*zeroth-order ADO*).
* **$\text{vec}(\mathrm{ADO}_k)$ (untere Blöcke):** Die vektorisierten Hilfsoperatoren (*Auxiliary Density Operators*), die jeweils ebenfalls $d \times d$-Matrizen darstellen und die quantenmechanischen Bad-Korrelationen speichern.
* **Gesamtdimension:** Da $\mathcal{N}_{\mathrm{ADOs}}$ Matrizen der Dimension $d \times d$ untereinander gestapelt sind, beträgt die Gesamtlänge des Vektors $\mathcal{D}_{\mathrm{tot}} = \mathcal{N}_{\mathrm{ADOs}} \cdot d^2$.

Auf diesem erweiterten Raum ist die Zeitevolution exakt und lokal in der Zeit (formal Markovsch) und gehorcht dem linearen Differentialgleichungssystem:

$$\frac{\mathrm{d}}{\mathrm{d}t} \vec{\Sigma}(t) = \mathcal{L}_{\mathrm{HEOM}} \vec{\Sigma}(t) \tag{7.1}$$

wobei der zeitunabhängige HEOM-Liouville-Superoperator $\mathcal{L}_{\mathrm{HEOM}}$ eine Matrix der Größe $\mathcal{D}_{\mathrm{tot}} \times \mathcal{D}_{\mathrm{tot}}$ darstellt. Die Zeitentwicklung über einen festen Zeitschritt $\Delta t$ lässt sich exakt über das Matrixexponential ausdrücken:

$$P = \exp(\mathcal{L}_{\mathrm{HEOM}} \Delta t) \in \mathbb{C}^{\mathcal{D}_{\mathrm{tot}} \times \mathcal{D}_{\mathrm{tot}}} \implies \vec{\Sigma}(t+\Delta t) = P \ @ \ \vec{\Sigma}(t)$$

Für eine Zeitentwicklung um $n$ Schritt muss man dann einfach den propagator $P$ $n$-mal auf den Startzustand bei $t=0$ anwenden:

$$\vec{\Sigma}(n\Delta t) = P^n\,\vec{\Sigma}(0),\qquad \vec{\Sigma}(0) = \begin{pmatrix}\mathrm{vec}(\rho_S(0))\\ \mathbf{0}\end{pmatrix} \tag{7.2}$$

> **Proposition 18.** Auf dem erweiterten Raum $\mathbb{C}^{\mathcal{D}_{\mathrm{tot}}}$ bildet die diskrete HEOM-Zeitentwicklung eine Halbgruppe: Der Zustand $\vec{\Sigma}_n = P\,\vec{\Sigma}_{n-1}$ hängt ausschließlich vom unmittelbaren Vorgänger ab.

**Beweis.** Da der HEOM-Liouvillian $\mathcal{L}_{\mathrm{HEOM}}$ zeitunabhängig ist, gilt für das Matrixexponential die Standard-Halbgruppeneigenschaft $\mathrm{e}^{\mathcal{L}(t+s)} = \mathrm{e}^{\mathcal{L}t}\,\mathrm{e}^{\mathcal{L}s}$ und somit $P(t+s) = P(t)P(s)$ bzw.$P^{n+m} = P^n P^m$. Für diskrete Zeitschritte folgt damit unmittelbar:

$$\vec{\Sigma}(n\Delta t) = \mathrm{e}^{\mathcal{L}n\Delta t}\vec{\Sigma}(0) = \mathrm{e}^{\mathcal{L}\Delta t}\,\mathrm{e}^{\mathcal{L}(n-1)\Delta t}\vec{\Sigma}(0) = \mathrm{e}^{\mathcal{L}\Delta t}\, \vec{\Sigma}\big((n-1)\Delta t\big) = P\,\vec{\Sigma}\big((n-1)\Delta t\big). \quad \blacksquare$$

Das ist eine reine Ein-Schritt-Rekursion: Zur Bestimmung von $\vec{\Sigma}_n$ genügt allein der Zustand $\vec{\Sigma}_{n-1}$ aus dem letzten Schritt – frühere Zeitschritte werden nicht benötigt. Im gegensatz zu $E$ entsteht somit auf dem Gesamtraum (System + Bad) per Konstruktion kein Gedächtnis-Rest ($R_n = 0$).

## Das neue Hindernis: $\Vert P\Vert_2 = 32{,}39$

Proposition 13 aus Kapitel 5 lässt sich direkt auf diesen Fall übertragen: Wir müssen lediglich $E$ durch den Propagator $P$ und $X_0$ durch den erweiterten Vektor $\vec{\Sigma}_0$ ersetzen. Da der Beweis rein auf dem Teleskopprodukt basiert und keine speziellen Eigenschaften von $E$ voraussetzt, gilt für die Gesamterfolgswahrscheinlichkeit $p_{\mathrm{total}}$ hier mit $\vec{\Sigma}_0$ und $P$ exakt dieselbe Zerlegung:

$$p_{\mathrm{total}} = \underbrace{\frac{1}{s^{2n}}}_{\text{Teil 1 (Gatter-Strafe)}}\cdot\underbrace{\frac{\Vert P^n\vec{\Sigma}_0\Vert^2}{\Vert\vec{\Sigma}_0\Vert^2}}_{\text{Teil 2 (physikalische Längenänderung)}},\qquad s = \Vert P\Vert_2 \tag{7.3}$$

Während bei Route B der Teil 2 harmlos war und Teil 1 das Problem darstellte, offenbart sich hier ein noch lehrreicheres Phänomen: **Beide Faktoren nehmen absurde Werte an** — und zwar in entgegengesetzte Richtungen.

**Teil 1: Die Gatter-Strafe.** Für unser konkretes Modell liefert die numerische Auswertung leider einen sehr kleinen Wert:

$$\Vert P\Vert_2 = 32{,}39 \quad\implies\quad \text{Teil 1} = \frac{1}{32{,}39^{200}} \approx 8{,}17\times10^{-303}$$

**Teil 2: Die Längenänderung.** Physikalisch würde man erwarten, dass dissipative Prozesse die Zustände dämpfen und dieser Faktor $\le 1$ bleibt. Entlang der tatsächlichen Trajektorie *wächst* die standardmäßige euklidische Länge des Vektors jedoch drastisch an, was unsere Erfolgswahrscheinlichkeit erhöht:

| $n$ | $0$ | $1$ | $2$ | $5$ | $10$ | $25$ | $50$ | $100$ |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| $\Vert P^n\vec\Sigma_0\Vert/\Vert\vec\Sigma_0\Vert$ | $1$ | $1{,}98$ | $19{,}8$ | $432$ | $1872$ | $2950$ | $2717$ | $1101$ |

Das Maximum der relativen Norm erreicht sogar einen Wert von $3165$. Für $n = 100$ Zeitschritte ergibt sich damit $\text{Teil 2} = 1101^2 \approx 1{,}21\times10^{6}$. 

Kombiniert man beide Terme, reproduziert dies exakt die gemessene winzige Erfolgswahrscheinlichkeit:

$$p_{\mathrm{total}}(100) = \underbrace{8{,}17\times10^{-303}}_{\text{Teil 1}}\cdot\underbrace{1{,}21\times10^{6}}_{\text{Teil 2}} \approx 9{,}90\times10^{-297}$$

## 7.3 Die Eichung: ein Skalarprodukt, in dem $\mathcal{L}$ dissipativ ist

Genau das machen wir mit $P$. Nur können wir die richtigen Koordinaten in $2640$ Dimensionen nicht raten — wir müssen sie ausrechnen. Das Werkzeug dafür ist die Lyapunov-Gleichung. 

Im Folgenden zeigen wir zuerst, dass 
$$W := \int_0^\infty \Big(\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\Big)^\dagger\,\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\,\mathrm d\tau = \int_0^\infty \mathrm{e}^{-2\delta\tau}\,\mathrm{e}^{\mathcal{L}^\dagger\tau}\,\mathrm{e}^{\mathcal{L}\tau}\,\mathrm d\tau$$
alle gewünschten Eigenschaften (Hermitizität und positive Definitheit) erfüllt und eine valide Lösung der Lyapunov-Gleichung ist. Da $W$ hermitesch ($W = W^\dagger$) und strikt positiv definit ist, garantiert der Spektralsatz eine unitäre Diagonalisierung der Form $W = V D V^\dagger = \sum_{i} w_i v_i v_i^\dagger$ mit orthonormalen Eigenvektoren $V$ ($V^\dagger V = \mathbb{I}$) und strikt positiven reellen Eigenwerten $w_i > 0$. Über den Funktionalkalkül definieren wir die eindeutige positive Matrixwurzel $W^{1/2}$ sowie deren Inverse $W^{-1/2}$ direkt über die Spektraldarstellung:

$$W^{1/2} := V D^{1/2} V^\dagger = V \begin{pmatrix} \sqrt{w_1} & & 0 \\ & \ddots & \\ 0 & & \sqrt{w_n} \end{pmatrix} V^\dagger$$

$$W^{-1/2} := (W^{1/2})^{-1} = V D^{-1/2} V^\dagger = V \begin{pmatrix} \frac{1}{\sqrt{w_1}} & & 0 \\ & \ddots & \\ 0 & & \frac{1}{\sqrt{w_n}} \end{pmatrix} V^\dagger$$

Diese Matrizen bilden die gesuchte Gauge-Transformation $\widetilde{P} = W^{1/2} P W^{-1/2}$, welche die Normkontraktion im transformierten Koordinatensystem erzwingt.

Anschließend zeigen wir, dass die Lyapunov-Gleichung unter diesen Bedingungen eindeutig lösbar ist und $W$ somit die einzige Lösung darstellt. Schlussendlich nutzen wir die Lyapunov-Gleichung, um eine Schranke für die Norm von $P$ herzuleiten. 

Dabei verwenden wir den spektralen Shift $\delta > 0$, um den Realteil der Eigenwerte von $\mathcal{L}$ strikt in die linke Halbebene zu verschieben: Sei $v$ ein Eigenvektor von $\mathcal{L}$ zum Eigenwert $\lambda$, also gilt per Definition $\mathcal{L} v = \lambda v$. Wenden wir nun die verschobene Matrix $(\mathcal{L} - \delta\mathbb{I})$ auf denselben Vektor $v$ an:

$$(\mathcal{L} - \delta\mathbb{I}) v = \mathcal{L} v - \delta\mathbb{I} v = \lambda v - \delta v = (\lambda - \delta) v$$

Das zeigt, dass der Vektor $v$ exakt derselbe Eigenvektor bleibt und der neue zugehörige Eigenwert zwingend $\lambda - \delta$ lautet. Da dies für jeden beliebigen Eigenvektor $v_i$ von $\mathcal{L}$ gilt, verschiebt sich ausnahmslos jeder Eigenwert $\lambda_i \to \lambda_i - \delta$.

> **Proposition 20 (Lyapunov-Eichung).** Sei $\delta>0$ so gewählt, dass $\mathcal{L}-\delta\mathbb{I}$ Hurwitz ist, d.h. $\mathrm{Re}\,\lambda < \delta$ für jeden Eigenwert $\lambda$ von $\mathcal{L}$. Dann besitzt die Lyapunov-Gleichung
> $$(\mathcal{L}-\delta\mathbb{I})^\dagger W + W(\mathcal{L}-\delta\mathbb{I}) = -\,\mathbb{I} \tag{7.5}$$
> genau eine Lösung $W$; diese ist hermitesch und positiv definit, und mit $\Vert x\Vert_W := \Vert W^{1/2}x\Vert_2$ gilt
> $$\Vert e^{\mathcal{L}\tau}\Vert_W \; := \; \Vert W^{1/2}e^{\mathcal{L}\tau}\Vert_2 \;\le\; e^{\delta\tau}\qquad\text{für alle }\tau\ge0 . \tag{7.6}$$

**Beweis.** Der Beweis gliedert sich in vier nachvollziehbare Schritte:

**(a) Wohldefiniertheit und Positivität des Lösungsansatzes:**  
Da $\delta\mathbb{I}$ mit $\mathcal{L}$ kommutiert, faktorisiert das Matrixexponential gemäß $\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau} = \mathrm{e}^{-\delta\tau}\mathrm{e}^{\mathcal{L}\tau}$. Wir definieren den Lösungsansatz (Kandidaten) für $W$ als Integral über alle zukünftigen Zustände der verschobenen Dynamik:

$$W := \int_0^\infty \Big(\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\Big)^\dagger\,\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\,\mathrm d\tau = \int_0^\infty \mathrm{e}^{-2\delta\tau}\,\mathrm{e}^{\mathcal{L}^\dagger\tau}\,\mathrm{e}^{\mathcal{L}\tau}\,\mathrm d\tau \tag{7.7}$$

Weil der Generator $\mathcal{L}-\delta\mathbb{I}$ Hurwitz ist (alle Eigenwert-Realteile sind echt negativ), zerfällt der Propagator $\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}$ für $\tau \to \infty$ exponentiell. Wenn die Matrix diagonalisierbar ist kann man das schön sehen:
$$A = V D V^{-1} = V \begin{pmatrix} \lambda_1 & 0 & \cdots \\ 0 & \lambda_2 & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix} V^{-1} \implies \mathrm{e}^{A\tau} = V \mathrm{e}^{D\tau} V^{-1} = V \begin{pmatrix} \mathrm{e}^{\lambda_1 \tau} & 0 & \cdots \\ 0 & \mathrm{e}^{\lambda_2 \tau} & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix} V^{-1} \xrightarrow{\tau \to \infty} \begin{pmatrix} 0 & 0 & \cdots \\ 0 & 0 & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix} = \mathbf{0}$$
Es gilt jedoch auch wenn die Matrix nicht diagonalisierbar ist. Somit existieren Konstanten $c,\eta>0$ mit $\Vert \mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\Vert_2 \le c\,\mathrm{e}^{-\eta\tau}$. Der Integrand erhält in der 2-Norm durch $c^2\mathrm{e}^{-2\eta\tau}$ also eine obere Schranke, womit das Integral garantiert nicht divergiert und $W$ daher wohlbestimmt ist.

* **Hermitezität ($W = W^\dagger$):** Der Integrand hat für jedes $\tau$ die Gestalt $Z^\dagger Z$ und ist damit symmetrisch/hermitesch.  
* **Strikte positive Definitheit ($x^\dagger W x > 0$):** Für jeden Testvektor $x \neq 0$ summiert das Integral die Längenquadrate der Trajektorie auf:
  $$x^\dagger W x = \int_0^\infty \big\Vert \mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}x\big\Vert_2^2\,\mathrm d\tau > 0.$$
  Da der Integrand stetig und nichtnegativ ist und beim Startwert $\tau=0$ den strikt positiven Wert $\Vert x\Vert_2^2 > 0$ annimmt, ist die Gesamtsumme $x^\dagger W x$ echt größer als null. Die Bedingung $x^\dagger W x > 0$ ist gleichbedeutend damit, dass alle Eigenwerte echt positiv sind. Somit existiert die Wurzel $\sqrt{\lambda_i}$ im Reellen, und die Transformationsmatrix $W^{1/2} = V\,\mathrm{diag}(\sqrt{\lambda_i})\,V^\dagger$ ist vollständig invertierbar ohne Division durch null.

**(b) Der Ansatz löst die Lyapunov-Gleichung (7.5):**  
Wir betrachten den Integranden $G(\tau) := \big(\mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}\big)^\dagger \mathrm{e}^{(\mathcal{L}-\delta\mathbb{I})\tau}$ und bilden dessen Zeitableitung. Mit der Produktregel folgt unmittelbar:

$$\frac{\mathrm dG}{\mathrm d\tau} = (\mathcal{L}-\delta\mathbb{I})^\dagger G(\tau) + G(\tau)(\mathcal{L}-\delta\mathbb{I}).$$

Integrieren wir diese Beziehung auf beiden Seiten von $\tau = 0$ bis $\tau = \infty$, so liefert die linke Seite nach dem Hauptsatz der Differential- und Integralrechnung einen einfachen Randterm:

$$\int_0^\infty \frac{\mathrm dG}{\mathrm d\tau}\,\mathrm d\tau = \Big[G(\tau)\Big]_0^\infty = G(\infty) - G(0) = \mathbf{0} - \mathbb{I} = -\mathbb{I},$$

da $G(\infty) = \mathbf{0}$ (exponentieller Zerfall aus Schritt a) und $G(0) = \mathrm{e}^{\mathbf{0}} = \mathbb{I}$ gilt. Auf der rechten Seite ziehen wir die konstanten Generatormatrizen vor das Integral:

$$-\mathbb{I} = (\mathcal{L}-\delta\mathbb{I})^\dagger \underbrace{\int_0^\infty G(\tau)\,\mathrm d\tau}_{=\,W} + \underbrace{\int_0^\infty G(\tau)\,\mathrm d\tau}_{=\,W} (\mathcal{L}-\delta\mathbb{I}) \quad\implies\quad (\mathcal{L}-\delta\mathbb{I})^\dagger W + W(\mathcal{L}-\delta\mathbb{I}) = -\mathbb{I}.$$

Damit erfüllt $W$ exakt die kontinuierliche Lyapunov-Gleichung.

**(c) Eindeutigkeit der Lösung:**  
Um zu zeigen, dass es genau eine Matrix $W$ gibt, die die Lyapunov-Gleichung löst, fassen wir die linke Seite von (7.5) als lineare Abbildung auf dem Raum aller Matrizen auf (einen sogenannten *Sylvester-Operator*):

$$\mathcal{S}(X) := (\mathcal{L}-\delta\mathbb{I})^\dagger X + X(\mathcal{L}-\delta\mathbb{I})$$

Die Gleichung lautet damit kompakt $\mathcal{S}(W) = -\mathbb{I}$. Aus der linearen Algebra ist bekannt: Ein lineares Gleichungssystem besitzt genau dann für jede rechte Seite eine eindeutige Lösung, wenn der zugehörige Operator **keinen Eigenwert gleich null** hat (also invertierbar ist).

* **Woher kommen die Eigenwerte von $\mathcal{S}$?**  
  Der Operator $\mathcal{S}(X) = A^\dagger X + X A$ wirkt auf eine Matrix $X$ gleichzeitig von links (über $A^\dagger$) und von rechts (über $A$). 
  
  Wählt man als Testmatrix das äußere Produkt $X = u v^\dagger$ aus einem Eigenvektor $u$ von $A^\dagger$ (mit Eigenwert $\overline{\lambda_j}$, also $A^\dagger u = \overline{\lambda_j} u$) und einem Eigenvektor $v$ von $A$ (mit $A v = \lambda_i v \implies v^\dagger A = \lambda_i v^\dagger$), sieht man die Wirkung direkt:
  
  $$\begin{aligned}
  \mathcal{S}(u v^\dagger) 
  &= \big(A^\dagger u\big) v^\dagger + u \big(v^\dagger A\big) = \big(\overline{\lambda_j} u\big) v^\dagger + u \big(\lambda_i v^\dagger\big) = (\overline{\lambda_j} + \lambda_i)\,(u v^\dagger)
  \end{aligned}$$
  
  Die Matrix $X = u v^\dagger$ verhält sich also exakt wie ein Eigenvektor des Operators $\mathcal{S}$, und der zugehörige Eigenwert ist schlicht die Summe $\overline{\lambda_j} + \lambda_i$.
* **Warum kann kein Eigenwert null sein?**  
  Da $(\mathcal{L}-\delta\mathbb{I})$ Hurwitz ist, besitzen ausnahmslos alle Eigenwerte einen strikt negativen Realteil: $\mathrm{Re}(\lambda_k) < 0$. Für jede beliebige Summe zweier Eigenwerte gilt daher:
  $$\mathrm{Re}(\overline{\lambda_j} + \lambda_i) = \underbrace{\mathrm{Re}(\lambda_j)}_{<\,0} + \underbrace{\mathrm{Re}(\lambda_i)}_{<\,0} < 0 \quad\implies\quad \overline{\lambda_j} + \lambda_i \neq 0.$$

Da kein einziger Eigenwert von $\mathcal{S}$ auf der Null liegt, ist der Kern trivial ($\ker(\mathcal{S}) = \{\mathbf{0}\}$). Der Operator $\mathcal{S}$ ist bijektiv invertierbar, und die Lösung $W = \mathcal{S}^{-1}(-\mathbb{I})$ existiert und ist strikt eindeutig.

**(d) Herleitung der Normabschätzung (7.6):**  
In diesem letzten Schritt zeigen wir, wie die Wahl von $W$ garantiert, dass die Dynamik in der neuen Metrik nicht unkontrolliert anwachsen kann.

* **Die Idee (Lyapunov-Funktion als „Energie“):**  
  Wir betrachten das Längenquadrat des Zustandsvektors in der $W$-Metrik als eine verallgemeinerte Energie:
  $$V(\tau) := \Vert x(\tau)\Vert_W^2 = x(\tau)^\dagger W x(\tau)$$
  Ziel ist es zu prüfen, wie sich diese Energie entlang einer echten physikalischen Trajektorie $\dot{x} = \mathcal{L}x$ über die Zeit verändert.

* **Schritt 1: Zeitableitung von $V(\tau)$ bilden**  
  Mit der Produktregel für Skalarprodukte und dem Einsetzen der Bewegungsgleichung $\dot{x} = \mathcal{L}x$ (bzw. $\dot{x}^\dagger = x^\dagger \mathcal{L}^\dagger$) erhalten wir:
  $$\frac{\mathrm dV}{\mathrm d\tau} = \dot{x}^\dagger W x + x^\dagger W \dot{x} = x^\dagger \big(\mathcal{L}^\dagger W + W\mathcal{L}\big) x$$

* **Schritt 2: Die Lyapunov-Gleichung einsetzen**  
  Wir stellen die Lyapunov-Gleichung $(\mathcal{L}-\delta\mathbb{I})^\dagger W + W(\mathcal{L}-\delta\mathbb{I}) = -\mathbb{I}$ nach dem Term $\mathcal{L}^\dagger W + W\mathcal{L}$ um:
  $$\mathcal{L}^\dagger W + W\mathcal{L} = -\mathbb{I} + 2\delta W$$
  Eingesetzt in die Zeitableitung ergibt sich:
  $$\frac{\mathrm dV}{\mathrm d\tau} = x^\dagger \big(-\mathbb{I} + 2\delta W\big) x = \underbrace{-x^\dagger x}_{=\,-\Vert x\Vert_2^2} + 2\delta \underbrace{x^\dagger W x}_{=\,V(\tau)}$$

* **Schritt 3: Der dissipative Kern**  
  Der Term $-\Vert x\Vert_2^2$ ist für jeden Zustand $x \neq 0$ **strikt negativ**. Er wirkt wie eine permanente Reibung, die dem System Energie entzieht. Lässt man diesen negativen Verlustterm weg, erhält man eine obere Schranke:
  $$\frac{\mathrm dV}{\mathrm d\tau} = -\Vert x\Vert_2^2 + 2\delta V(\tau) \;\le\; 2\delta V(\tau)$$

* **Schritt 4: Integration über den Zeitverlauf**  
  Die Differentialungleichung $\frac{\mathrm dV}{\mathrm d\tau} \le 2\delta V(\tau)$ besagt, dass $V(\tau)$ höchstens so schnell wachsen kann wie eine Exponentialfunktion mit Rate $2\delta$. 
  
  Multipliziert man beide Seiten mit dem positiven Faktor $\mathrm{e}^{-2\delta\tau}$ und wendet die Produktregel rückwärts an, folgt:
  $$\frac{\mathrm d}{\mathrm d\tau} \Big(\mathrm{e}^{-2\delta\tau} V(\tau)\Big) \le 0 \quad\implies\quad \mathrm{e}^{-2\delta\tau} V(\tau) \le V(0) \quad\implies\quad V(\tau) \le \mathrm{e}^{2\delta\tau} V(0)$$

* **Schritt 5: Rückübersetzung auf die Operatornorm**  
  Ersetzen wir $V(\tau) = \Vert x(\tau)\Vert_W^2$ und $V(0) = \Vert x_0\Vert_W^2$ und ziehen die Quadratwurzel, erhalten wir:
  $$\Vert x(\tau)\Vert_W \le \mathrm{e}^{\delta\tau} \Vert x_0\Vert_W$$
  Da $x(\tau) = \mathrm{e}^{\mathcal{L}\tau} x_0$ für jeden beliebigen Startzustand $x_0$ gilt, folgt über das Supremum aller normierten Startvektoren unmittelbar die Schranke für die induzierte Operatornorm:
  $$\Vert \mathrm{e}^{\mathcal{L}\tau}\Vert_W = \sup_{x_0 \neq 0} \frac{\Vert \mathrm{e}^{\mathcal{L}\tau} x_0\Vert_W}{\Vert x_0\Vert_W} \le \mathrm{e}^{\delta\tau}. \qquad \blacksquare$$

### Was die Eichung mit dem Propagator macht

> **Proposition 21.** Mit $W$ aus Proposition 20 sei
> $$\tilde P := W^{1/2}\,P\,W^{-1/2},\qquad \tilde{\vec\Sigma}_0 := W^{1/2}\vec{\Sigma}_0 .$$
> Dann gilt
> $$\Vert\tilde P\Vert_2 \le e^{\delta\Delta\tau} \qquad\text{und}\qquad \tilde P^{\,n} = W^{1/2}P^{n}W^{-1/2} , \tag{7.8}$$
> insbesondere ist die Trajektorie $\tilde{\vec\Sigma}_n = \tilde P^{\,n}\tilde{\vec\Sigma}_0$ **exakt** und man liest daraus
> $$\mathrm{vec}\,\rho_S(n\Delta t) = Q\,W^{-1/2}\,\tilde{\vec\Sigma}_n \tag{7.9}$$
> ab. Die Eichung führt also **keinerlei** Näherung ein.

**Beweis.** Da die Wurzel $W^{1/2}$ invertierbar ist, können wir jeden Vektor $y$ eindeutig als $y = W^{1/2}x$ schreiben. Für die Norm substituieren wir daher im Supremum $y = W^{1/2}x$. Weil $W^{1/2}$ invertierbar ist, ist das eine Bijektion, das Supremum läuft also über dieselbe Menge:

$$\begin{aligned}
\Vert\tilde P\Vert_2 = \sup_{y\neq0}\frac{\Vert \tilde{P}y\Vert_2}{\Vert y\Vert_2} = \sup_{y\neq0}\frac{\Vert W^{1/2}PW^{-1/2}y\Vert_2}{\Vert y\Vert_2}
\;\overset{y=W^{1/2}x}{=}\; \sup_{x\neq0}\frac{\Vert W^{1/2}Px\Vert_2}{\Vert W^{1/2}x\Vert_2}
= \sup_{x\neq0}\frac{\Vert Px\Vert_W}{\Vert x\Vert_W}
= \Vert P\Vert_W = \Vert e^{\mathcal{L}\Delta\tau}\Vert_W \;\overset{(7.6)}{\le}\; e^{\delta\Delta\tau}
\end{aligned}$$

Die Potenz ist ein Teleskopprodukt, in dem sich $W^{-1/2}W^{1/2}=\mathbb{1}$ immer wieder aufhebt:

$$\tilde P^{\,n} = \big(W^{1/2}PW^{-1/2}\big)^n = W^{1/2}P\,\underbrace{W^{-1/2}W^{1/2}}_{=\,\mathbb{1}}\,P\,\underbrace{W^{-1/2}W^{1/2}}_{=\,\mathbb{1}}\,P\cdots P\,W^{-1/2} = W^{1/2}P^{n}W^{-1/2}$$

 Startet man im transformierten Bild bei $\tilde{\vec{\Sigma}}_0 = W^{1/2}\vec{\Sigma}_0$, lautet der Zustand nach $n$ Schritten:

  $$\tilde{\vec{\Sigma}}_n = \tilde{P}^n \tilde{\vec{\Sigma}}_0 = \big(W^{1/2}P^n W^{-1/2}\big)\big(W^{1/2}\vec{\Sigma}_0\big) = W^{1/2}\big(P^n\vec{\Sigma}_0\big) = W^{1/2}\vec{\Sigma}_n.$$

  Multipliziert man von links mit $W^{-1/2}$, erhält man den exakten physikalischen Zustand $\vec{\Sigma}_n = W^{-1/2}\tilde{\vec{\Sigma}}_n$ zurück. Die abschließende Projektion $Q$ liefert genau die gesuchte System-Dichtematrix $\rho_S(n\Delta\tau)$ gemäß (7.9). $\;\blacksquare$

Jetzt lesen wir Gleichung (7.3) noch einmal, aber in der neuen Eichung:
$$p_{\mathrm{total}} = \underbrace{\frac{1}{\Vert\tilde{P}\Vert_2^{2n}}}_{\text{Teil 1 (Gatter-Strafe)}} \cdot \underbrace{\frac{\Vert\tilde{P}^n \tilde{\vec{\Sigma}}_0\Vert_2^2}{\Vert\tilde{\vec{\Sigma}}_0\Vert_2^2}}_{\text{Teil 2 (Längenänderung)}} = \underbrace{\frac{1}{\Vert P\Vert_W^{2n}}}_{\text{Teil 1}} \cdot \underbrace{\frac{\Vert P^n \vec{\Sigma}_0\Vert_W^2}{\Vert\vec{\Sigma}_0\Vert_W^2}}_{\text{Teil 2}}$$

- Teil 1: Das Gatter wird aus $\tilde P$ gebaut, der Skalierungsfaktor ist also $s = \Vert\tilde P\Vert_2 \le e^{\delta\Delta\tau}$, und die Gatter-Strafe über $n$ Schritte wird zu $$s^{2n} \;\le\; \Big(e^{\delta\Delta\tau}\Big)^{2n} = e^{2\delta\,n\Delta\tau} = e^{2\delta T} \tag{7.10}$$ Man wählt $\delta$ so klein, dass $\delta T \ll 1$ ist, und Teil 1 aus (7.3) ist erledigt. 
- Teil 2 wird jetzt ebenfalls in der $W$-Norm gemessen, und dort ist es durch $e^{2\delta T}$ nach oben beschränkt: Nach Proposition 20 gilt für jeden Zeitschritt $\tau = n\Delta t = T$ und für jeden beliebigen Startvektor: $\Vert \mathrm{e}^{\mathcal{L}T} x_0\Vert_W \le \mathrm{e}^{\delta T} \Vert x_0\Vert_W$. Da $P^n \vec{\Sigma}_0 = \mathrm{e}^{\mathcal{L}n\Delta t} \vec{\Sigma}_0 = \mathrm{e}^{\mathcal{L}T}\vec{\Sigma}_0$ ist, setzen wir $x_0 = \vec{\Sigma}_0$ ein:$$\Vert P^n \vec{\Sigma}_0\Vert_W \le \mathrm{e}^{\delta T} \Vert \vec{\Sigma}_0\Vert_W$$Teilt man durch $\Vert \vec{\Sigma}_0\Vert_W$ und quadriert beide Seiten, erhält man direkt die obere Schranke für Teil 2:$$\text{Teil 2} = \frac{\Vert P^n \vec{\Sigma}_0\Vert_W^2}{\Vert \vec{\Sigma}_0\Vert_W^2} = \frac{\Vert\tilde{P}^n \tilde{\vec{\Sigma}}_0\Vert_2^2}{\Vert\tilde{\vec{\Sigma}}_0\Vert_2^2} \;\le\; \Big(\mathrm{e}^{\delta T}\Big)^2 = \mathrm{e}^{2\delta T}$$

Die Strafe hängt also **nur noch vom Produkt $\delta T$** ab, nicht mehr von der Zahl der Schritte (aber vom Zeithorizont $T$) und nicht von der Nicht-Normalität von $P$. 

**Warum darf $\delta$ nicht null sein?** Dass $\delta > 0$ gewählt werden muss, ist eine direkte physikalische Konsequenz des thermischen Gleichgewichts:
* Der Liouvillian $\mathcal{L}$ besitzt den Eigenwert $\lambda_0 = 0$, da der stationäre Gleichgewichtszustand $\vec{\Sigma}_{\mathrm{ss}}$ zeitunabhängig ist ($\mathcal{L}\vec{\Sigma}_{\mathrm{ss}} = 0 = 0 \cdot \vec{\Sigma}_{\mathrm{ss}}$).
* Ohne Shift ($\delta = 0$) würde das Integral zur Berechnung von $W = \int_0^\infty \mathrm{e}^{\mathcal{L}^\dagger\tau}\,\mathrm{e}^{\mathcal{L}\tau}\,\mathrm{d}\tau$ in Gleichung (7.7) entlang dieses stationären Zustands unendlich lange über eine konstante Funktion integrieren und somit **divergieren**.


Gemessen für unser Modell ($T = 1000\,\mathrm{fs}$, also $T = 0{,}1884\,\mathrm{cm}$): bei $\delta = 0{,}1$ fällt $\Vert\tilde P^{\,n}\tilde{\vec\Sigma}_0\Vert/\Vert\tilde{\vec\Sigma}_0\Vert$ über 100 Schritte auf $0{,}455$, Teil 2 ist also $0{,}21$. Genau das ist der Punkt der Eichung: Sie macht **beide** Faktoren einzeln zu Zahlen der Ordnung $1$, statt zwei astronomische Zahlen gegeneinander wegkürzen zu lassen.

| $\delta$ [cm$^{-1}$] | $\Vert\tilde P\Vert_2$ | Schranke $e^{\delta\Delta\tau}$ | $\mathrm{cond}(W)$ | $p_{\mathrm{total}}(100)$ |
| ---: | ---: | ---: | ---: | ---: |
| $0{,}02$ | $1{,}000038$ | $1{,}000038$ | $1{,}74\times10^{10}$ | $0{,}365$ |
| $0{,}1$ | $1{,}000188$ | $1{,}000188$ | $1{,}68\times10^{10}$ | $0{,}200$ |
| $0{,}5$ | $1{,}000942$ | $1{,}000942$ | $1{,}62\times10^{10}$ | $0{,}129$ |
| $2{,}0$ | $1{,}003774$ | $1{,}003774$ | $1{,}42\times10^{10}$ | $0{,}062$ |
| $10{,}0$ | $1{,}019015$ | $1{,}019015$ | $7{,}48\times10^{9}$ | $0{,}0037$ |

Zwei Dinge fallen auf. Erstens ist die Schranke (7.6) **scharf** — die gemessene Norm trifft $e^{\delta\Delta\tau}$ auf sechs Stellen. Zweitens ändert sich $\mathrm{cond}(W)$ über einen Faktor $500$ in $\delta$ hinweg kaum; sie wird also nicht vom Shift bestimmt, sondern von der Spannweite der ADO-Skalen im HEOM-Ansatz. $\delta$ ist damit ein freier Parameter, den man für lange Läufe einfach kleiner macht.

Zum Vergleich derselbe Propagator, ungeeicht und geeicht:

$$\Vert P\Vert_2 = 32{,}39 \quad\longrightarrow\quad \Vert\tilde P\Vert_2 = 1{,}000038 \qquad\text{und}\qquad p_{\mathrm{total}}(100):\quad 10^{-301}\;\longrightarrow\;0{,}365$$

## 7.4 Jetzt wird die Arnoldi-Kompression gratis

$\tilde P$ operiert noch immer auf $\mathbb{C}^{2640}$ ($\lceil\log_2 2640\rceil + 1 = 13$ Qubits). Wir komprimieren daher wie zuvor via Arnoldi-Verfahren, nun jedoch auf dem geeichten Propagator: Mit dem Krylov-Raum $\tilde{\mathcal{K}}_m := \mathcal{K}_m(\tilde P, \tilde{\vec\Sigma}_0)$, der Orthonormalbasis $\tilde Q_m$ und der komprimierten Matrix $\tilde H_m := \tilde Q_m^\dagger \tilde P \tilde Q_m$ gilt:

> **Proposition 22.** Für **jedes** $m$ gilt: $\Vert \tilde H_m\Vert_2 \le \Vert\tilde P\Vert_2 \le \mathrm{e}^{\delta\Delta\tau} \approx 1$.

**Beweis.** Da $\tilde Q_m$ eine Isometrie ist, folgt $\Vert \tilde H_m\Vert_2 \le \Vert \tilde P\Vert_2$ direkt aus den Eigenschaften der orthogonalen Projektion. Die zweite Schranke liefert Proposition 21. $\;\blacksquare$

Damit entfällt der fundamentale Trade-off bisheriger Routen:
* **Bisher (z. B. Route B):** Ein zu großes $m$ nahm unphysikalische Richtungen auf; $\Vert H_m\Vert_2$ stieg gegen die nutzlose Schranke $\sqrt{2}$, was die Erfolgswahrscheinlichkeit zerstörte (z. B. $p \sim 10^{-18}$ bei $m=64$).
* **Mit Kontraktions-Eichung:** Weil die obere Schranke nun $\Vert\tilde H_m\Vert_2 \le 1{,}000038$ lautet, bleibt $\tilde H_m$ für **beliebiges** $m$ eine Kontraktion. 

Die Krylov-Dimension $m$ ist somit kein heikler Kompromissparameter mehr, sondern ein **reiner Konvergenzparameter**: Man kann $m$ bedenkenlos so lange erhöhen, bis die gewünschte Genauigkeit erreicht ist. $H_m$ sit dasnn eine $m \times m$ Matrix, was $\lceil\log_2 m\rceil$ Qubits auf dem circuit kostet.

| $m$ | Qubits | $\Vert\tilde H_m\Vert_2$ | Abweichung von QuTiP | $p_{\mathrm{total}}(100)$ |
| ---: | ---: | ---: | ---: | ---: |
| $16$ | $5$ | $1{,}000038$ | $1{,}93\times10^{-1}$ | $0{,}172$ |
| $32$ | $6$ | $1{,}000038$ | $6{,}22\times10^{-2}$ | $0{,}332$ |
| $64$ | $7$ | $1{,}000038$ | $\mathbf{5{,}27\times10^{-7}}$ | $0{,}365$ |
| $96$ | $8$ | $1{,}000038$ | $5{,}27\times10^{-7}$ | $0{,}365$ |
| $128$ | $8$ | $1{,}000038$ | $5{,}27\times10^{-7}$ | $0{,}365$ |

Zwei Aspekte sind hierbei zentral:

Erstens stagniert die Abweichung zu QuTiP ab $m = 64$ bei einem Plateau von $5{,}27 \times 10^{-7}$.  Genau diese Eigenschaft fehlte ohne Eichung: dort wäre $p_{\mathrm{total}}$ ab einem gewissen $m$ wieder eingebrochen. Dies stellt keinen Approximationsfehler der Arnoldi-Kompression dar, sondern markiert die numerische Integrationstoleranz des QuTiP-Referenzsolvers.

Um die tatsächliche Konvergenz der Arnoldi-Kompression ohne Solver-Artefakte offenzulegen, vergleicht man die komprimierte Dynamik stattdessen direkt mit der exakten, unkomprimierten Referenztrajektorie. Betrachtet man einen langen Zeithorizont von $n = 500$ Zeitschritten (bei $\delta = 0{,}1\,\mathrm{cm}^{-1}$), fällt dieser reine Kompressionsfehler streng monoton ab:
* $m = 32$: $3{,}15 \times 10^{-1}$
* $m = 64$: $2{,}91 \times 10^{-2}$
* $m = 96$: $1{,}97 \times 10^{-4}$
* $m = 128$: $1{,}65 \times 10^{-8}$
* $m = 192$: $3{,}35 \times 10^{-14}$

Es tritt keinerlei Instabilität oder Fehlerumkehr bei großen Krylov-Dimensionen auf.

**Registergröße.** Von $\mathcal{D}_{\mathrm{tot}} = 2640$ (12 System-Qubits) schrumpfen wir auf $m = 128$ (7 System-Qubits), zusammen mit der Ancilla also **8 Qubits**. Route B kam mit $m=32$ auf 6 Qubits, ist also etwas sparsamer — dafür aber um vier Größenordnungen ungenauer und braucht 25 klassisch gerechnete Zeitschritte. Bei $m=64$ liegt Route C mit 7 Qubits schon bei $5{,}27\times10^{-7}$.

## 7.5 Das Gitter und die Ablesung

Die $m\times m$-Matrix $\tilde H_m$ wird auf $m_p = 2^{\lceil\log_2 m\rceil}$ aufgefüllt und **einmal** nach Kapitel 5 dilatiert:

$$U = \begin{pmatrix} \tilde H_m/s & B\\ C & -(\tilde H_m/s)^\dagger\end{pmatrix},\qquad B = \sqrt{\mathbb{1}-\tfrac{\tilde H_m\tilde H_m^\dagger}{s^2}},\qquad C = \sqrt{\mathbb{1}-\tfrac{\tilde H_m^\dagger\tilde H_m}{s^2}},\qquad s = \Vert\tilde H_m\Vert_2$$

Danach besteht der gesamte Schaltkreis aus genau einem Gatter, das man wiederholt:

```
krylov (7 Qubits) : ────[ U ]─────────────[ U ]─────────────[ U ]────────── ... ───> Y_n
                         │                 │                 │
Ancilla           : |0>──[ U ]─(M)──|0>────[ U ]─(M)──|0>────[ U ]─(M)─|0>─ ...
                               │                 │                 │
klass. Register   : ───────────●─────────────────●─────────────────●─────── ...
                             rec[1]            rec[2]            rec[3]
```

Zwei Unterschiede zum Bild in dem Abschnitt *Die Umsetzung auf dem Quantencomputer* fallen auf.

**Erstens brauchen wir keine Zustandspräparation.** Der Startvektor des Krylov-Raums ist per Konstruktion $q_1 = \tilde{\vec\Sigma}_0/\Vert\tilde{\vec\Sigma}_0\Vert$, in Krylov-Koordinaten also $y_0 = \Vert\tilde{\vec\Sigma}_0\Vert\,e_1$. Das Register startet damit ohnehin schon im richtigen Zustand $\vert 0\cdots0\rangle$; die gesamte Information über $\rho_S(0)$ und die Eichung steckt in der klassischen Zahl $\Vert\tilde{\vec\Sigma}_0\Vert$ und in der Basis $\tilde Q_m$.

**Zweitens messen wir die Ancilla sofort nach jedem Schritt und setzen sie zurück**, statt für jeden Schritt ein frisches Ancilla-Qubit zu spendieren. Nach dem Prinzip der aufgeschobenen Messung, das wir im Abschnitt *Many steps* diskutiert haben, ist das exakt äquivalent: Sobald Schritt $t$ vorbei ist, berührt kein Gatter dieses Ancilla-Qubit je wieder, die Messung kommutiert also mit allem Folgenden. Der Vorteil ist, dass wir mit **einem** Ancilla-Qubit für beliebig viele Schritte auskommen, statt $n$ davon zu brauchen. Das Messprotokoll wandert stattdessen in das klassische Register `rec`.

### Der akzeptierte Zweig ist deterministisch

> **Proposition 24.** Bedingt auf das Messprotokoll $\mathtt{rec} = 0\cdots0$ ist der Zustand des Krylov-Registers nach $t$ Schritten
> $$\vert Y_t\rangle = \frac{\tilde H_m^{\,t}\,y_0}{\Vert\tilde H_m^{\,t}\,y_0\Vert} \tag{7.12}$$
> und damit **unabhängig von jedem Zufall in den Messungen**.

**Beweis.** Induktion über $t$. Für $t=0$ ist nichts zu zeigen. Sei das Register vor Schritt $t$ im reinen Produktzustand $\vert Y_{t-1}\rangle\otimes\vert0\rangle_E$. Die Anwendung von $U$ spaltet ihn nach dem Abschnitt *One step* in genau zwei Zweige auf:

$$U\big(\vert0\rangle_E\otimes\vert Y_{t-1}\rangle\big) = \vert0\rangle_E\otimes\Big(\tfrac{\tilde H_m}{s}Y_{t-1}\Big) \;+\; \vert1\rangle_E\otimes\big(C\,Y_{t-1}\big)$$

Die Messung wählt zufällig einen der beiden Zweige aus — welchen, das ist die einzige Zufälligkeit im ganzen Ablauf. *Gegeben* das Ergebnis $\vert0\rangle$ projiziert der Kollaps aber auf den ersten Summanden, und die Normierung, die die Natur automatisch vornimmt, hebt den Faktor $1/s$ wieder auf:

$$\vert Y_t\rangle = \frac{\big(\tilde H_m/s\big)Y_{t-1}}{\big\Vert\big(\tilde H_m/s\big)Y_{t-1}\big\Vert} = \frac{\tfrac1s\,\tilde H_mY_{t-1}}{\tfrac1s\,\Vert\tilde H_mY_{t-1}\Vert} = \frac{\tilde H_m\,Y_{t-1}}{\Vert\tilde H_m\,Y_{t-1}\Vert}$$

Das ist eine **Funktion von $Y_{t-1}$ allein**, ohne jeden Rest an Zufall. Das anschließende `reset` bringt die Ancilla wieder nach $\vert0\rangle$, so dass die Induktionsvoraussetzung für Schritt $t+1$ wiederhergestellt ist. Setzt man die Rekursion $t$-mal ein und benutzt $y_0 = \Vert\tilde{\vec\Sigma}_0\Vert\,e_1 \propto Y_0$, folgt (7.12). $\;\blacksquare$

Das hat eine sehr praktische Konsequenz. Der Zufall im Schaltkreis steckt *ausschließlich* in der Frage, **ob** ein Durchlauf akzeptiert wird, nicht darin, **was** man im Erfolgsfall vorfindet. Ein einziger akzeptierter Shot trägt daher bereits den exakten Zustand; die Statistik über viele Shots wird nur für die skalare Zahl $p_{\mathrm{total}}$ gebraucht.

### Die Rückrechnung auf die Physik

Der Quantencomputer gibt uns den normierten Vektor $Y_t$. Um daraus $\rho_S(t)$ zu machen, brauchen wir zwei Dinge: seine Länge und die Rücktransformation aus den Krylov- und Eichkoordinaten. Die Länge liefert Proposition 13, wörtlich übertragen mit $E\to\tilde H_m$:

$$p_{\mathrm{total}}(t) = \frac{\Vert\tilde H_m^{\,t}y_0\Vert^2}{s^{2t}\Vert y_0\Vert^2} \quad\Longrightarrow\quad \Vert y_t\Vert \;=\; \underbrace{\Vert\tilde{\vec\Sigma}_0\Vert}_{=\,\Vert y_0\Vert}\cdot\, s^{\,t}\,\sqrt{p_{\mathrm{total}}(t)} \tag{7.13}$$

Die Rücktransformation fassen wir in einer einzigen klassischen Matrix zusammen, die **einmal** vorab berechnet wird:

$$R := \Big(W^{-1/2}\,\tilde Q_m\Big)_{[\,1:d^2,\;:\,]} \;\in\;\mathbb{C}^{d^2\times m} \qquad\Longrightarrow\qquad \mathrm{vec}\,\rho_S(t\Delta t) = R\;y_t \tag{7.14}$$

Denn nach (7.9) ist $\mathrm{vec}\rho_S = \Pi \, W^{-1/2}\tilde{\vec\Sigma}_t$, nach der Kompression ist $\tilde{\vec\Sigma}_t \approx \tilde Q_m y_t$, und $\Pi$ schneidet genau die ersten $d^2$ Zeilen heraus. Zusammengesetzt:

$$\mathrm{vec}\,\rho_S(t\Delta t) = \Pi \underbrace{\,W^{-1/2}\tilde Q_m}_{=\;R}\;\underbrace{\Vert\tilde{\vec\Sigma}_0\Vert\,s^{\,t}\sqrt{p_{\mathrm{total}}(t)}}_{=\;\lambda_t}\;\cdot\;Y_t$$

| Größe | Bedeutung | Woher kommt der Wert? | Wann im Ablauf? |
| :--- | :--- | :--- | :--- |
| $\Vert\tilde{\vec\Sigma}_0\Vert$ | Länge des geeichten Startvektors | $\Vert W^{1/2}\vec\Sigma_0\Vert$ | **vor** dem Quanten-Run, rein klassisch |
| $s$ | Dilatations-Skalierung | $s = \Vert\tilde H_m\Vert_2 \le e^{\delta\Delta\tau}$ | **vor** dem Quanten-Run, rein klassisch |
| $R$ | Rücktransformation in den Systemblock | $R = (W^{-1/2}\tilde Q_m)_{[1:d^2,:]}$ | **vor** dem Quanten-Run, rein klassisch |
| $p_{\mathrm{total}}(t)$ | kumulierte Erfolgswahrscheinlichkeit | Anteil der Shots mit $\mathtt{rec}[1..t] = 0\cdots0$ | **nach** dem Run, aus dem klassischen Register |
| $Y_t$ | normierter Registerzustand | Zustand der $\lceil\log_2 m\rceil$ Krylov-Qubits, bedingt auf die akzeptierten Shots | am Zeitschritt $t$ |

**Eine zweite, unabhängige Ablesung.** Es gibt einen Weg, der ohne $p_{\mathrm{total}}$ auskommt und somit das statistische Rauschen (Shot Noise) der Erfolgszählung eliminiert, was das ergebnis genauer macht. Die HEOM-Dynamik erhält die Spur des nullten ADOs, es gilt also zu jeder Zeit exakt $\mathrm{Tr}\,\rho_S(t) = 1$. Das ist keine Zusatzinformation über die Lösung, sondern eine Eigenschaft der Bewegungsgleichung, und sie legt den fehlenden Skalar fest:

$$\rho_S(t) = \frac{\mathrm{unvec}\big(R\,Y_t\big)}{\mathrm{Tr}\,\mathrm{unvec}\big(R\,Y_t\big)}$$

**Beweis:** Der unnormierte Vektor $y_t$ und der vom Quantencomputer ausgegebene, normierte Zustand $Y_t$ unterscheiden sich nur durch einen reellen, positiven Skalierungsfaktor $\lambda_t > 0$:$$y_t = \lambda_t \cdot Y_t \qquad \text{mit } \lambda_t = \Vert\tilde{\vec{\Sigma}}_0\Vert \cdot s^t \sqrt{p_{\mathrm{total}}(t)}$$Wendet man die lineare Rücktransformationsmatrix $R$ an, gilt wegen der Linearität:$$\operatorname{vec}(\rho_S(t)) = R \, y_t = R \, (\lambda_t Y_t) = \lambda_t \cdot (R Y_t)$$Wenden wir nun die lineare Operation $\operatorname{unvec}$ (das Umformen des $d^2$-Vektors in eine $d \times d$-Matrix) und anschließend die Spur $\operatorname{Tr}$ an:$$\operatorname{Tr}\big(\rho_S(t)\big) = \operatorname{Tr}\Big(\operatorname{unvec}\big(\lambda_t \cdot R Y_t\big)\Big) = \lambda_t \cdot \operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big)$$Da die physikalische Dichtematrix zu jedem Zeitpunkt strikt spurerhaltend ist ($\operatorname{Tr}(\rho_S(t)) = 1$), folgt zwingend:$$1 = \lambda_t \cdot \operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big) \quad\implies\quad \lambda_t = \frac{1}{\operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big)}$$ Setzt man diesen Skalar $\lambda_t$ nun oben wieder ein, bekommt man: $$\rho_S(t) = \lambda_t \cdot \operatorname{unvec}(R Y_t) = \left(\frac{1}{\operatorname{Tr}\Big(\operatorname{unvec}(R Y_t)\Big)}\right) \cdot \operatorname{unvec}(R Y_t) \, \qquad \blacksquare$$

Nebenbei erledigt das auch die globale Phase, die ein normierter Quantenzustand ohnehin nur bis auf $e^{i\varphi}$ festlegt: Die physikalische Spur muss reell und positiv sein. Beide Ablesungen benutzen **disjunkte** Information — die eine das Messprotokoll, die andere die Spurerhaltung —, ihre Übereinstimmung ist also ein in sich geschlossener Test des ganzen Kapitels.

## Zusammenfassung des Kapitels

Der Weg in einem Absatz: Die HEOM-Hierarchie ist auf dem erweiterten Raum bereits eine Halbgruppe, $\vec\Sigma_n = P^n\vec\Sigma_0$ ist exakt (Proposition 18), und der ganze Gedächtnis-Rest $R_n$ aus Route B entsteht erst dadurch, dass man mit $Q$ auf den Systemblock projiziert und die ADOs wegwirft. Wer sie behält, braucht kein $K$ und keine klassisch vorberechnete Historie. Das einzige Hindernis ist dann, dass $P$ mit $\rho(P)=1$ zwar brav, mit $\Vert P\Vert_2 = 32{,}39$ aber stark nicht-normal ist, und dass die Sz.-Nagy-Dilatation stur die Operatornorm berechnet — ein Hindernis, das auch feinere Zeitschritte nicht beseitigen (Proposition 19). Die Auflösung ist, dass $32{,}39$ eine Aussage über das *Skalarprodukt* ist: Die Lyapunov-Gleichung (7.5) liefert eine Metrik $W$, in der $P$ eine Kontraktion bis auf $e^{\delta\Delta\tau}$ ist (Proposition 20), ohne die Trajektorie im Geringsten zu verändern (Proposition 21). In dieser Eichung wird die Arnoldi-Kompression zum reinen Konvergenzparameter (Propositionen 22 und 23), und der akzeptierte Zweig des Schaltkreises ist deterministisch (Proposition 24).

$$\boxed{\;\Vert P\Vert_2 = 32{,}39 \;\xrightarrow{\;W^{1/2}\cdot W^{-1/2}\;}\; \Vert\tilde P\Vert_2 = 1{,}000038 \qquad\Longrightarrow\qquad p_{\mathrm{total}}(100):\;\;10^{-301}\;\longrightarrow\;0{,}37\;}$$

| Proposition | Aussage |
| :---: | :--- |
| **18** | HEOM ist auf dem ADO-Raum eine Halbgruppe; $R_n \equiv 0$ ohne jedes $K$ |
| **20** | Lyapunov-Eichung: $\exists\,W\succ0$ mit $\Vert e^{\mathcal{L}\tau}\Vert_W \le e^{\delta\tau}$ |
| **21** | $\tilde P = W^{1/2}PW^{-1/2}$ ist ähnlich zu $P$ — die Eichung ist exakt |
| **22** | $\Vert\tilde H_m\Vert_2 \le e^{\delta\Delta\tau}$ für jedes $m$ — Kompression bleibt Kontraktion |
| **24** | der auf $0\cdots0$ bedingte Zweig ist deterministisch — ein Shot genügt für den Zustand |

Die Implementierung liegt in `heom_gauge.py` (Erzeuger, Eichung, Arnoldi, Dilatation) und `gauge_circuit.py` (das Gitter); die Rechnungen und alle Abbildungen stehen in `main_durchgaengiges_grid.ipynb`.

---
## 7.6 Woher die schlechte Kondition wirklich kommt

Die Eichung hat eine Schwachstelle, die man erst bei starker Kopplung sieht:
$\mathrm{cond}(W)$ wird riesig, und irgendwann reicht `float64` nicht mehr. Bei
$\lambda = 100\,\mathrm{cm}^{-1}$ messen wir $\mathrm{cond}(W) = 2{,}25\times10^{19}$,
und dann liefert die Rechnung $\Vert\tilde P\Vert_2 = 7{,}15$ statt der von
Proposition 20 **garantierten** $\le 1{,}000038$. Die Garantie kann mathematisch
nicht verletzt werden — es ist also die Arithmetik, die aufgibt.

Es lohnt sich, zwei Fragen zu trennen: *warum* ist $\mathrm{cond}(W)$ so groß,
und *warum* bringt uns das um?

### Warum $\mathrm{cond}(W)$ groß ist — und dass das kein Zufall ist

$W$ hat genau eine Aufgabe: den Raum so zu verzerren, dass aus einem Operator mit
$\Vert P\Vert_2 = 7{,}7\times10^5$ eine Kontraktion wird. Eine Metrik, die ein
transientes Wachstum um den Faktor $7{,}7\times10^5$ platt bügeln soll, muss den
Raum selbst um ungefähr diesen Faktor dehnen. Also ist

$$\mathrm{cond}(W)\;\gtrsim\;\Vert P\Vert_2^{\,2}$$

keine Schwäche des Verfahrens, sondern eine untere Schranke des *Problems*.
Gemessen an drei Punkten passt das gut:

$$\mathrm{cond}(W) \;\approx\; 3{,}8\times10^{7}\cdot\Vert P\Vert_2^{2}$$

| $\lambda$ [cm$^{-1}$] | $\Vert P\Vert_2$ | $\mathrm{cond}(W)$ gemessen | Modell |
| ---: | ---: | ---: | ---: |
| $3$ | $32{,}4$ | $1{,}74\times10^{10}$ | $4{,}0\times10^{10}$ |
| $35$ | $3{,}12\times10^{4}$ | $5{,}15\times10^{16}$ | $3{,}7\times10^{16}$ |
| $100$ | $7{,}71\times10^{5}$ | $2{,}25\times10^{19}$ | $2{,}3\times10^{19}$ |

Damit ist klar, wo man ansetzen kann: **entweder am Symptom** (sorgfältiger
rechnen, damit $\mathrm{cond}(W)$ länger erträglich bleibt) **oder an der
Ursache** ($\Vert P\Vert_2$ kleiner machen). Es stellt sich heraus, dass der
zweite Weg der ungleich bessere ist.

### Die Ursache: die ADOs sind gar nicht gleich groß

Warum wächst $\Vert P\Vert_2$ überhaupt so, obwohl $\rho(P) = 1$ ist? Weil die
ADOs in der üblichen HEOM-Konvention völlig verschiedene Größenordnungen haben.
Der Hilfsoperator zum Multi-Index $\mathbf n = (n_1,\dots,n_K)$ trägt den
Vorfaktor $\prod_k c_k^{n_k}$, und die Badkoeffizienten $c_k$ — Drude-Term und
Padé-Terme — unterscheiden sich um Größenordnungen. Ein „Einheitsschritt" in
einem tiefen ADO bedeutet numerisch also etwas ganz anderes als ein
Einheitsschritt in $\rho_S$.

Das ist, als würde man ein Zimmer in Millimetern und eine Erdumlaufbahn in
denselben Koordinaten führen und sich dann wundern, dass die Abbildung
zwischen beiden riesige Zahlen enthält. Das transiente Wachstum von $P$ ist
genau das: der Propagator schiebt Gewicht von kleinskaligen in großskalige
Koordinaten, und die euklidische Norm zählt das als Wachstum mit.

Die Standardabhilfe ist die Umskalierung nach Shi et al.:

$$\tilde\rho_{\mathbf n} \;=\; \frac{\rho_{\mathbf n}}{\sqrt{\prod_k n_k!\,|c_k|^{n_k}}}
\qquad\Longleftrightarrow\qquad
\tilde{\mathcal L} = D^{-1}\mathcal L\,D,\quad D = \mathrm{diag}(s_{\mathbf n})$$

Das ist eine **Diagonal-Ähnlichkeitstransformation**, also exakt — an der
Dynamik ändert sich nichts, nur an den Koordinaten. Und weil der nullte ADO das
Label $(0,\dots,0)$ und damit den Faktor $s = 1$ trägt, sind Anfangszustand und
Ablesung von der Umskalierung überhaupt nicht betroffen.

Der Effekt ist drastisch:

| $\lambda$ [cm$^{-1}$] | $\Vert P\Vert_2$ roh | $\Vert P\Vert_2$ **umskaliert** | Faktor | Spreizung $s_{\max}/s_{\min}$ |
| ---: | ---: | ---: | ---: | ---: |
| $3$ | $32{,}39$ | $\mathbf{1{,}001}$ | $32$ | $1{,}08\times10^{5}$ |
| $35$ | $3{,}12\times10^{4}$ | $\mathbf{1{,}017}$ | $3{,}1\times10^{4}$ | $4{,}18\times10^{6}$ |
| $100$ | $7{,}71\times10^{5}$ | $\mathbf{1{,}023}$ | $7{,}5\times10^{5}$ | $2{,}07\times10^{7}$ |

Die „Normkatastrophe" aus Abschnitt 7.2 ist damit zu einem großen Teil ein
**Artefakt der Koordinatenwahl**, nicht der Physik. Nach der Umskalierung ist
$P$ schon fast von selbst eine Kontraktion, und die Lyapunov-Eichung muss nur
noch den letzten Rest von $1{,}02$ auf $1{,}00004$ erledigen — mit einem $W$,
dessen Kondition nach obigem Modell um $\Vert P\Vert^2$, also um bis zu elf
Größenordnungen, kleiner ausfällt.

Zwei Anmerkungen dazu:

* **QuTiP macht das nicht.** Das `renorm`-Argument gab es bis QuTiP 4.6 und ist
  ersatzlos entfallen; `solver.rhs(0)` liefert den unskalierten Erzeuger. Die
  Umskalierung ist also tatsächlich noch zu holen und nicht schon eingebaut.
* **`matrix_balance` ist kein Ersatz.** Es sucht blind eine Diagonale, die
  Zeilen- und Spaltennormen angleicht, und kam bei $\lambda = 3$ nur von
  $32{,}4$ auf $2{,}54$ — bei $\lambda = 100$ bewirkte es sogar exakt gar nichts
  ($\Vert P_{\rm bal}\Vert = \Vert P\Vert$). Die physikalische Skalierung kennt
  die richtige Diagonale, weil sie weiß, was die Koordinaten *bedeuten*.

### Das Symptom: $W$ zu bilden quadriert die Kondition

Auch mit kleinerem $\mathrm{cond}(W)$ bleibt die Frage, warum ausgerechnet
$10^{16}$ die Grenze ist. Der Grund liegt im Rechenweg. Wir lösen erst nach $W$
und ziehen dann mit `eigh` die Wurzel. `eigh` löst Eigenwerte aber nur bis zu
einer **absoluten** Genauigkeit $\varepsilon\,\lambda_{\max}$ auf — alles, was
kleiner ist als $\lambda_{\max}\cdot10^{-16}$, ist Rauschen. Bei
$\mathrm{cond}(W) = 2{,}25\times10^{19}$ liegt $\lambda_{\min}$ vier
Größenordnungen *unter* dieser Grenze, und $W^{-1/2}$ wird aus Müll gebaut.

Der Ausweg ist, dass wir $W$ gar nicht brauchen. Für eine Eichung genügt
**irgendein** $G$ mit $G^\dagger G = W$, denn

$$\Vert x\Vert_W^2 \;=\; x^\dagger W x \;=\; x^\dagger G^\dagger G x \;=\; \Vert Gx\Vert_2^2 .$$

Die symmetrische Wurzel $W^{1/2}$ ist nur eine mögliche Wahl. **Hammarlings
Verfahren** (SLICOT `SB03OD`) löst die Lyapunov-Gleichung direkt aus der
Schur-Form von $A$ nach dem Cholesky-Faktor $G$, ohne $W$ je zu bilden. Weil
$\mathrm{cond}(G) = \sqrt{\mathrm{cond}(W)}$ ist, wird die Dynamik nie
quadriert, und die Grenze verschiebt sich von $\mathrm{cond}(W)\sim10^{16}$ auf
$\sim10^{32}$:

| Rechenweg | Grenze in $\mathrm{cond}(W)$ | erlaubtes $\Vert P\Vert_2$ |
| :--- | ---: | ---: |
| $W$ bilden, `eigh`-Wurzel | $10^{16}$ | $1{,}6\times10^{4}$ |
| Hammarling (Faktor direkt) | $10^{32}$ | $1{,}6\times10^{12}$ |

Das deckt sich mit der Beobachtung, dass $\lambda = 35$ mit
$\Vert P\Vert = 3{,}1\times10^4$ die Garantie gerade verletzt.

Eine praktische Hürde bleibt: **SLICOT rechnet reell, der HEOM-Erzeuger ist
komplex.** Man kann das komplexe Problem reell einbetten,

$$\tilde A = \begin{pmatrix}\mathrm{Re}\,A & -\,\mathrm{Im}\,A\\ \mathrm{Im}\,A & \mathrm{Re}\,A\end{pmatrix},
\qquad x^\dagger W x = \tilde x^{\mathsf T}\tilde W\tilde x,\quad \tilde x = \begin{pmatrix}\mathrm{Re}\,x\\ \mathrm{Im}\,x\end{pmatrix},$$

und bekommt einen reellen Faktor auf $\mathbb R^{2n}$. Der ist eine gültige
Eichung für die *reelle* Darstellung, lässt sich aber nicht ohne Weiteres in ein
komplexes $n\times n$ zurückschreiben — der Cholesky-Faktor einer strukturierten
Matrix ist selbst nicht strukturiert. Das ganze Gitter auf die reelle Darstellung
umzustellen wäre möglich (und würde ein Qubit kosten), ist aber nicht gemacht:
Nach der ADO-Umskalierung liegt $\Vert P\Vert_2$ ohnehin bei $\approx1{,}02$,
und damit ist die Frage für den gemessenen Kopplungsbereich gegenstandslos.
`lyapunov_cholesky_factor()` in `heom_gauge.py` stellt das Verfahren als
Diagnose bereit.

### $\delta$: kleiner ist besser, als ich zunächst dachte

Der Shift $\delta$ sieht nach einem reinen Schönheitsfehler aus — er kostet
$e^{-2\delta T}$ und muss nur größer als null sein. Man würde daraus schließen,
dass $\delta$ kaum eine Rolle spielt, solange $\delta T \ll 1$ ist. Die Messung
sagt etwas anderes:

| $\delta$ [cm$^{-1}$] | $\mathrm{cond}(W)$ | $\Vert\tilde P\Vert_2$ | $e^{-2\delta T}$ | Abweichung | $p_{\mathrm{total}}(100)$ |
| ---: | ---: | ---: | ---: | ---: | ---: |
| $10^{-1}$ | $1{,}68\times10^{10}$ | $1{,}000188$ | $0{,}963$ | $5{,}27\times10^{-7}$ | $0{,}1995$ |
| $10^{-3}$ | $2{,}89\times10^{11}$ | $1{,}000002$ | $0{,}9996$ | $5{,}27\times10^{-7}$ | $0{,}8896$ |
| $\mathbf{10^{-7}}$ | $2{,}87\times10^{15}$ | $1{,}000000$ | $1{,}0000$ | $5{,}27\times10^{-7}$ | $\mathbf{1{,}0000}$ |
| $10^{-11}$ | $2{,}14\times10^{19}$ | $3{,}56$ | $1{,}0000$ | $5{,}28\times10^{-7}$ | $0{,}0000$ |

Bei $\delta = 10^{-7}$ ist die Erfolgswahrscheinlichkeit **exakt 1** — die
Post-Selection kostet gar nichts mehr. Der Faktor $e^{-2\delta T}$ erklärt das
nicht (er ändert sich nur von $0{,}963$ auf $1{,}000$), und genau daran habe ich
mich zuerst verrechnet. Der wahre Grund ist, dass mit $\delta$ auch $W$ *selbst*
sich ändert: Je kleiner $\delta$, desto stärker gewichtet die Metrik den
stationären Zustand — und der Zustand relaxiert ja *auf ihn zu*. In dieser
Metrik verliert die Trajektorie also kaum noch Länge, und $p_{\mathrm{total}}$
geht gegen eins.

Umsonst ist auch das nicht: Der interessante, zerfallende Anteil wird damit zu
einem immer kleineren Bruchteil der Registerlänge, und auf echter Hardware
kostet genau das Shots beim Auslesen. In `float64` ist die Genauigkeit über den
ganzen Bereich unverändert $5{,}27\times10^{-7}$, aber die Schranke nach unten
ist hart: zwischen $10^{-7}$ und $10^{-11}$ reißt $\mathrm{cond}(W)$ die
$10^{16}$-Marke, $\Vert\tilde P\Vert$ springt auf $3{,}56$, und
$p_{\mathrm{total}}$ fällt auf null.

**Die brauchbare Faustregel** ist damit nicht „$\delta$ so klein wie möglich",
sondern: $\delta$ so klein wie möglich, *solange der Selbsttest aus Proposition
20 noch hält* — also solange die gemessene Norm $\Vert\tilde P\Vert_2$ die
garantierte Schranke $e^{\delta\Delta\tau}$ nicht überschreitet. Genau dafür ist
der Selbsttest da: Er sagt einem, wann man zu weit gegangen ist.